In [6]:
# inspect_ckpt_full.py
# === EDIT THIS PATH IF NEEDED ===
CKPT_PATH = r"C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\yolov8_sm_latest.pt"

import torch
import pprint
import re

print(f"[i] Loading checkpoint with weights_only=False (trusted file):\n    {CKPT_PATH}")
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

# --- 1) Top-level keys ---
print("\n[i] Top-level keys in checkpoint:")
print(list(ckpt.keys()))

# --- 2) Common Ultralytics / training metadata fields ---
meta_fields = ("train_args", "data", "yaml", "epoch", "date", "version", "git", "ema")
for k in meta_fields:
    if k in ckpt:
        print(f"\n== {k} ==")
        pprint.pprint(ckpt[k])

# --- 3) Try to extract info from the 'model' entry (could be object or dict) ---
m = ckpt.get("model", None)
if m is not None:
    print("\n[i] Found 'model' entry inside checkpoint.")

    # Try attribute-style fields (when 'model' is a nn.Module-like object)
    for attr in ("yaml", "args", "names", "nc"):
        try:
            if hasattr(m, attr):
                print(f"\n== model.{attr} ==")
                pprint.pprint(getattr(m, attr))
        except Exception as e:
            print(f"[i] Couldn't read model.{attr}: {e}")

    # Try dict-style fields (when 'model' is a plain dict)
    if isinstance(m, dict):
        for subk in ("yaml", "args", "names", "nc"):
            if subk in m:
                print(f"\n== model['{subk}'] ==")
                pprint.pprint(m[subk])

# --- 4) Deep scan for path-like strings anywhere in the checkpoint ---
def scan_for_paths(obj, path="root"):
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            hits += scan_for_paths(v, f"{path}.{k}")
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            hits += scan_for_paths(v, f"{path}[{i}]")
    elif isinstance(obj, str):
        # Find dataset YAMLs, image/label folders, file paths, etc.
        if re.search(r"(data.*\.ya?ml)|(/images?/)|(/labels?/)|(\.(jpg|png)\b)|([A-Za-z]:\\\\)", obj, re.IGNORECASE):
            hits.append((path, obj))
    return hits

hits = scan_for_paths(ckpt)
if hits:
    print("\n== Possible data/path hints found ==")
    for pth, s in hits:
        print(f"{pth}: {s}")
else:
    print("\n[i] No obvious data/path strings found in checkpoint.")

# --- 5) (Optional) Use Ultralytics to read names/nc cleanly, if installed ---
try:
    from ultralytics import YOLO
    print("\n[i] Ultralytics available: loading via YOLO(...) for class info")
    mdl = YOLO(CKPT_PATH)
    names = getattr(mdl, "names", None) or getattr(mdl.model, "names", None)
    nc = getattr(mdl.model, "nc", None)
    print("== YOLO loader info ==")
    print("Class names:", names)
    print("Num classes:", nc)
except Exception as e:
    print("\n[i] Ultralytics loader not available (this is optional).")
    print("    To enable: pip install ultralytics")
    print("    Error:", e)

print("\n[i] Done.\n"
      "Interpretation:\n"
      " - If you see 'data.yaml' or dataset folder paths that match your data, that's strong evidence of training overlap.\n"
      " - Matching 'names'/'nc' is consistent but not proof.\n"
      " - If nothing shows, this checkpoint likely omitted training metadata.")


[i] Loading checkpoint with weights_only=False (trusted file):
    C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\yolov8_sm_latest.pt

[i] Top-level keys in checkpoint:
['epoch', 'best_fitness', 'model', 'ema', 'updates', 'optimizer', 'train_args', 'train_metrics', 'train_results', 'date', 'version']

== train_args ==
{'agnostic_nms': False,
 'amp': True,
 'augment': False,
 'batch': 16,
 'box': 7.5,
 'cache': False,
 'cfg': None,
 'classes': None,
 'close_mosaic': 35,
 'cls': 0.5,
 'conf': None,
 'copy_paste': 0.0,
 'cos_lr': False,
 'data': '/home/participant/ai-coral-reefs2/yolov8/repo2/data/05_model_input/yolov8/v2/SEAFLOWER_BOLIVAR_and_SEAFLOWER_COURTOWN_and_SEAVIEW_ATL_and_SEAVIEW_IDN_PHL_and_SEAVIEW_PAC_AUS_and_TETES_PROVIDENCIA/data.yaml',
 'degrees': 45,
 'deterministic': True,
 'device': None,
 'dfl': 1.5,
 'dnn': False,
 'dropout': 0.0,
 'dynamic': False,
 'epochs': 100,
 'exist_ok': False,
 'fliplr': 0.5,
 'flipud': 0.5,
 'format': 'torchscript',
 'fraction': 1.0,
 'freeze': N

In [2]:
from ultralytics import YOLO
m = YOLO(r"C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\yolov8_sm_latest.pt")
print(m.names)  # expect something like {0: 'hard', 1: 'soft'} or {0:'soft',1:'hard'}


{0: 'hard_coral', 1: 'soft_coral'}


In [3]:
from ultralytics import YOLO
import torch, os, json

WEIGHTS = r"C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\yolov8_sm_latest.pt"

m = YOLO(WEIGHTS)

print("== Device ==")
print("CUDA available:", torch.cuda.is_available())
print("Using:", "cuda" if torch.cuda.is_available() else "cpu")

print("\n== Classes ==")
print(m.names)           # e.g., {0: 'hard', 1: 'soft'}
print("num_classes:", len(m.names))

print("\n== Model summary ==")
m.info(detailed=False)   # overview of layers, params

# Parameter count (total & trainable)
total = sum(p.numel() for p in m.model.parameters())
trainable = sum(p.numel() for p in m.model.parameters() if p.requires_grad)
print(f"\nParameters: total={total:,}  trainable={trainable:,}")

# Stride (useful for image size multiples)
print("\nStride:", int(m.model.stride.max()))


== Device ==
CUDA available: False
Using: cpu

== Classes ==
{0: 'hard_coral', 1: 'soft_coral'}
num_classes: 2

== Model summary ==
YOLOv8s-seg summary: 151 layers, 11,790,870 parameters, 0 gradients, 40.2 GFLOPs

Parameters: total=11,790,870  trainable=0

Stride: 32


In [5]:
from ultralytics import YOLO
import numpy as np

IMG = r"C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Datasets\mask_labels\content\gdrive\MyDrive\Data Challenge 3 - JBG060 AY2526\01_data\benthic_datasets\mask_labels\reef_support\SEAFLOWER_BOLIVAR\images\20220912_AnB_CB10 (42).JPG"

m = YOLO(WEIGHTS)
res = m.predict(source=IMG, imgsz=640, conf=0.25, iou=0.7, verbose=False)[0]

print("Image shape:", res.orig_shape)
print("Detections:", len(res.boxes or []))

if res.boxes is not None:
    cls_ids = res.boxes.cls.cpu().numpy().astype(int)
    counts = {m.names[c]: int((cls_ids == c).sum()) for c in np.unique(cls_ids)}
    print("Per-class instance counts (prediction):", counts)

# Access masks for further analysis/visualization
if res.masks is not None:
    masks = res.masks.data.cpu().numpy()   # (N,H,W), boolean/float
    print("Masks array shape:", masks.shape)


Image shape: (3000, 4000)
Detections: 32
Per-class instance counts (prediction): {'hard_coral': 27, 'soft_coral': 5}
Masks array shape: (32, 480, 640)


In [10]:
import torch

checkpoint_path = r"C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\yolov8_sm_latest.pt"

# allow loading full checkpoint
ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

print("Top-level keys:", ckpt.keys())

if "train_args" in ckpt:
    print("Training args:", ckpt["train_args"])


Top-level keys: dict_keys(['epoch', 'best_fitness', 'model', 'ema', 'updates', 'optimizer', 'train_args', 'train_metrics', 'train_results', 'date', 'version'])
Training args: {'task': 'segment', 'mode': 'train', 'model': 'yolov8s-seg.pt', 'data': '/home/participant/ai-coral-reefs2/yolov8/repo2/data/05_model_input/yolov8/v2/SEAFLOWER_BOLIVAR_and_SEAFLOWER_COURTOWN_and_SEAVIEW_ATL_and_SEAVIEW_IDN_PHL_and_SEAVIEW_PAC_AUS_and_TETES_PROVIDENCIA/data.yaml', 'epochs': 100, 'patience': 50, 'batch': 16, 'imgsz': 1024, 'save': True, 'save_period': -1, 'cache': False, 'device': None, 'workers': 8, 'project': 'data/06_models/yolov8/segment/', 'name': 'current_best_small', 'exist_ok': False, 'pretrained': True, 'optimizer': 'auto', 'verbose': True, 'seed': 0, 'deterministic': True, 'single_cls': False, 'rect': False, 'cos_lr': False, 'close_mosaic': 35, 'resume': False, 'amp': True, 'fraction': 1.0, 'profile': False, 'freeze': None, 'overlap_mask': True, 'mask_ratio': 4, 'dropout': 0.0, 'val': True

In [11]:
import os
import json
import pandas as pd

root_dir = r"C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Datasets\mask_labels\content\gdrive\MyDrive\Data Challenge 3 - JBG060 AY2526\01_data\benthic_datasets\mask_labels\reef_support"

def parse_ndjson(ndjson_path):
    records = []
    with open(ndjson_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line.strip())
            data_row = row.get("data_row", {})
            projects = row.get("projects", {})

            # Handle projects/labels
            if projects:
                for proj_id, proj_data in projects.items():
                    project_name = proj_data.get("name", "")
                    for label in proj_data.get("labels", []):
                        annotations = label.get("annotations", {}).get("objects", [])
                        if annotations:
                            for ann in annotations:
                                records.append({
                                    "dataset": data_row.get("details", {}).get("dataset_name", ""),
                                    "image_id": data_row.get("external_id", ""),
                                    "image_url": data_row.get("row_data", ""),
                                    "label_project": project_name,
                                    "class_name": ann.get("name", ""),
                                    "mask_url": ann.get("mask", {}).get("url", "")
                                })
                        else:
                            # case: image exists but no annotations
                            records.append({
                                "dataset": data_row.get("details", {}).get("dataset_name", ""),
                                "image_id": data_row.get("external_id", ""),
                                "image_url": data_row.get("row_data", ""),
                                "label_project": project_name,
                                "class_name": None,
                                "mask_url": None
                            })
            else:
                # fallback if no project/labels at all
                records.append({
                    "dataset": data_row.get("details", {}).get("dataset_name", ""),
                    "image_id": data_row.get("external_id", ""),
                    "image_url": data_row.get("row_data", ""),
                    "label_project": None,
                    "class_name": None,
                    "mask_url": None
                })
    return pd.DataFrame(records)

# Walk through all subfolders and process ndjson files
for root, _, files in os.walk(root_dir):
    for file in files:
        if file.endswith(".ndjson"):
            ndjson_path = os.path.join(root, file)
            print(f"📂 Processing {ndjson_path}")

            df = parse_ndjson(ndjson_path)

            # save CSV next to the original file
            csv_path = ndjson_path.replace(".ndjson", ".csv")
            df.to_csv(csv_path, index=False)

            print(f"✅ Saved CSV: {csv_path}, rows = {len(df)}")


📂 Processing C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Datasets\mask_labels\content\gdrive\MyDrive\Data Challenge 3 - JBG060 AY2526\01_data\benthic_datasets\mask_labels\reef_support\SEAFLOWER_BOLIVAR\export-result.ndjson
✅ Saved CSV: C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Datasets\mask_labels\content\gdrive\MyDrive\Data Challenge 3 - JBG060 AY2526\01_data\benthic_datasets\mask_labels\reef_support\SEAFLOWER_BOLIVAR\export-result.csv, rows = 2579
📂 Processing C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Datasets\mask_labels\content\gdrive\MyDrive\Data Challenge 3 - JBG060 AY2526\01_data\benthic_datasets\mask_labels\reef_support\SEAFLOWER_COURTOWN\export-result.ndjson
✅ Saved CSV: C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Datasets\mask_labels\content\gdrive\MyDrive\Data Challenge 3 - JBG060 AY2526\01_data\benthic_datasets\mask_labels\reef_support\SEAFLOWER_COURTOWN\export-result.csv, rows = 2449
📂 Processing C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Datasets\mask_labels\content\

In [20]:
import pandas as pd

df = pd.read_csv(r'C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Datasets\mask_labels\content\gdrive\MyDrive\Data Challenge 3 - JBG060 AY2526\01_data\benthic_datasets\mask_labels\reef_support\UNAL_BLEACHING_TAYRONA\export-result.csv')

df.columns

In [21]:
df.columns

Index(['dataset', 'image_id', 'image_url', 'label_project', 'class_name',
       'mask_url'],
      dtype='object')

In [26]:
df.head()

,dataset,image_id,image_url,label_project,class_name,mask_url
0,UNAL_Tayrona_Blanqueamiento,C1_BC_PM_T1_29nov24_CDaza_corr.jpg,https://storage.labelbox.com/clggufbis038y07y2...,UNAL_BLEACHING_TAYRONA,SCALE,NaN
1,UNAL_Tayrona_Blanqueamiento,C1_BC_PM_T1_29nov24_CDaza_corr.jpg,https://storage.labelbox.com/clggufbis038y07y2...,UNAL_BLEACHING_TAYRONA,Hard Coral,https://api.labelbox.com/api/v1/projects/cm6t6...
2,UNAL_Tayrona_Blanqueamiento,C1_BC_PM_T1_29nov24_CDaza_corr.jpg,https://storage.labelbox.com/clggufbis038y07y2...,UNAL_BLEACHING_TAYRONA,Hard Coral,https://api.labelbox.com/api/v1/projects/cm6t6...
3,UNAL_Tayrona_Blanqueamiento,C1_BC_PM_T1_29nov24_CDaza_corr.jpg,https://storage.labelbox.com/clggufbis038y07y2...,UNAL_BLEACHING_TAYRONA,Hard Coral,https://api.labelbox.com/api/v1/projects/cm6t6...
4,UNAL_Tayrona_Blanqueamiento,C1_BC_PM_T1_29nov24_CDaza_corr.jpg,https://storage.labelbox.com/clggufbis038y07y2...,UNAL_BLEACHING_TAYRONA,Hard Coral,https://api.labelbox.com/api/v1/projects/cm6t6...


In [24]:
class_counts = df["class_name"].value_counts()
print(class_counts)


class_name
Hard Coral                     5847
SCALE                           654
Soft Coral                      477
Milleporid                      372
Other Sessile Invertebrates      48
Name: count, dtype: int64


In [ ]:
import pandas as pd
# Keep only Hard Coral and Soft Coral
df_filtered = df[df["class_name"].isin(["Hard Coral", "Soft Coral"])]

print("Counts after filtering:")
print(df_filtered["class_name"].value_counts())

# Save to new CSV
df_filtered.to_csv("export_result_hard_soft.csv", index=False)
print("✅ Saved filtered dataset to export_result_hard_soft.csv")


In [3]:
import pandas as pd
pd.read_csv(r'C:\Users\User\PycharmProjects\Cap_coral_reefs\New_version_of_the_project\output_with_labels.csv')


,type,name,name_ext,path,label_name,label_name_ext,label_path
0,mask,20220912_AnB_CB10 (103)_mask,20220912_AnB_CB10 (103)_mask.png,C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Da...,NaN,NaN,NaN
1,mask,20220912_AnB_CB10 (104)_mask,20220912_AnB_CB10 (104)_mask.png,C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Da...,NaN,NaN,NaN
2,mask,20220912_AnB_CB10 (110)_mask,20220912_AnB_CB10 (110)_mask.png,C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Da...,NaN,NaN,NaN
3,mask,20220912_AnB_CB10 (111)_mask,20220912_AnB_CB10 (111)_mask.png,C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Da...,NaN,NaN,NaN
4,mask,20220912_AnB_CB10 (113)_mask,20220912_AnB_CB10 (113)_mask.png,C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Da...,NaN,NaN,NaN
...,...,...,...,...,...,...,...
5960,mask,C9_PB_PSA_T3_19nov24_HBenavides_Corr_mask,C9_PB_PSA_T3_19nov24_HBenavides_Corr_mask.png,C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Da...,NaN,NaN,NaN
5961,mask,C9_PB_PSb_T1_19nov24_HBenavides_Corr_mask,C9_PB_PSb_T1_19nov24_HBenavides_Corr_mask.png,C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Da...,NaN,NaN,NaN
5962,mask,C9_PB_PSb_T2_19nov24_HBenavides_Corr_mask,C9_PB_PSb_T2_19nov24_HBenavides_Corr_mask.png,C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Da...,NaN,NaN,NaN
5963,mask,C9_PB_PSb_T3_19nov24_HBenavides_Corr_mask,C9_PB_PSb_T3_19nov24_HBenavides_Corr_mask.png,C:\Users\User\Desktop\YEAR 3\Q1\Capstone DC\Da...,NaN,NaN,NaN


In [4]:
# run this in Python inside your venv
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")


CUDA available: False
GPU name: —


In [1]:
from pathlib import Path

root = Path(r"C:\Users\User\Downloads\website\website\benthic_datasets\mask_labels\reef_support")

count_all = 0
for p in root.rglob("*"):
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]:
        count_all += 1

print("Total images found (anywhere under root):", count_all)


Total images found (anywhere under root): 61155


In [13]:
from pathlib import Path

root = Path(r"C:\Users\User\Downloads\website\website\benthic_datasets\mask_labels\reef_support")

for ds in sorted([d for d in root.iterdir() if d.is_dir()]):
    labels_dir = ds / "labels"
    if labels_dir.exists():
        count = len(list(labels_dir.glob("*.txt")))
        print(f"{ds.name}: {count} label files")
    else:
        print(f"{ds.name}: no labels folder")


out_fold: no labels folder
SEAFLOWER_BOLIVAR: 246 label files
SEAFLOWER_COURTOWN: 241 label files
SEAVIEW_ATL: 659 label files
SEAVIEW_IDN_PHL: 466 label files
SEAVIEW_PAC_AUS: 658 label files
SEAVIEW_PAC_USA: 278 label files
TETES_PROVIDENCIA: 105 label files
UNAL_BLEACHING_TAYRONA: 658 label files


In [14]:
total = 0
for ds in sorted([d for d in root.iterdir() if d.is_dir()]):
    labels_dir = ds / "labels"
    if labels_dir.exists():
        total += len(list(labels_dir.glob("*.txt")))
print("Total .txt label files:", total)


Total .txt label files: 3311


In [18]:
import pandas as pd
df = pd.read_csv(r'C:\Users\User\Downloads\website\website\benthic_datasets\mask_labels\reef_support\out_fold\output.csv')
df.head()

,type,name,name_ext,path,label_name,label_name_ext,label_path
0,mask,20220912_AnB_CB10 (103)_mask,20220912_AnB_CB10 (103)_mask.png,website/benthic_datasets/mask_labels/reef_supp...,NaN,NaN,NaN
1,mask,20220912_AnB_CB10 (104)_mask,20220912_AnB_CB10 (104)_mask.png,website/benthic_datasets/mask_labels/reef_supp...,NaN,NaN,NaN
2,mask,20220912_AnB_CB10 (110)_mask,20220912_AnB_CB10 (110)_mask.png,website/benthic_datasets/mask_labels/reef_supp...,NaN,NaN,NaN
3,mask,20220912_AnB_CB10 (111)_mask,20220912_AnB_CB10 (111)_mask.png,website/benthic_datasets/mask_labels/reef_supp...,NaN,NaN,NaN
4,mask,20220912_AnB_CB10 (113)_mask,20220912_AnB_CB10 (113)_mask.png,website/benthic_datasets/mask_labels/reef_supp...,NaN,NaN,NaN


In [4]:
from pathlib import Path
root = Path(r"C:\Users\User\Downloads\website\website\benthic_datasets\mask_labels\reef_support")
total = 0
for ds in sorted([d for d in root.iterdir() if d.is_dir()]):
    labels_dir = ds / "images"
    if labels_dir.exists():
        total += len(list(labels_dir.glob("*.jpg")))
print("Total .txt label files:", total)


Total .txt label files: 3311
